In [ ]:
import pandas as pd

In [ ]:
fp = "../data/sba_loans_prepared/sba_loans_risk_good_train.csv"
df = pd.read_csv(fp)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
clfrn = RandomForestClassifier(n_estimators=250, random_state=42)

In [ ]:
preds = df.columns.tolist()
preds.remove("LoanStatus")

In [ ]:
X_df = df[preds]
Y_rn = df.LoanStatus
#clfrn.fit(X_df, Y_rn)

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
calibrated_clf = CalibratedClassifierCV(clfrn, method="sigmoid", cv=5)
calibrated_clf.fit(X_df, Y_rn)

In [ ]:
clfrn.fit(X_df, Y_rn)
importances = clfrn.feature_importances_

In [ ]:
feature_importance_series = pd.Series(importances, index=X_df.columns)
import matplotlib.pyplot as plt

# Sort and plot top N features (optional)
top_n = 10 # Plot the top 5 most important features
feature_importance_series.nlargest(top_n).plot(kind='barh')
plt.title('Top Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.grid(True)
plt.show()

In [ ]:
pp_rn = Y_rn.value_counts()[1]/Y_rn.value_counts()[0]

In [ ]:
fp = "../data/sba_loans_prepared/sba_loans_num_enc_val.csv"
dfv = pd.read_csv(fp)
X_dfv = dfv[preds]
Y_v = dfv["LoanStatus"]

In [ ]:
val_res = {"prob_chgoff": calibrated_clf.predict_proba(X_dfv)[:, 1],
                "prob_PIF": calibrated_clf.predict_proba(X_dfv)[:, 0],
               "LoanStatus": Y_v}
df_val_res = pd.DataFrame.from_dict(val_res, orient="columns")

In [ ]:
df_val_res.LoanStatus.value_counts()

In [ ]:
from sklearn.metrics import precision_recall_curve,auc

In [ ]:
y_scores = calibrated_clf.predict_proba(X_dfv)[:, 1]

In [ ]:
Y_v

In [ ]:
# Calculate precision, recall, and thresholds
precision, recall, thresholds = precision_recall_curve(Y_v, y_scores)

In [ ]:
TN = 100
pTN = precision[:TN]
rTN = recall[:TN]
tTN = thresholds[:TN]
df_th_res = pd.DataFrame.from_dict({"thresh": tTN, "precision": pTN, "recall": rTN}, orient="columns")

In [ ]:
df_th_res.tail(20)

In [ ]:
THSEL = 0.002945

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.plot(thresholds, precision[:-1], 'b-', label='Precision', marker="x")
plt.plot(thresholds, recall[:-1], 'r-', label='Recall', marker="o")
plt.title("Precision-Recall on Validation Dataset")
plt.xlabel('Threshold')
plt.legend(loc='lower left')
plt.ylim([0,1])
plt.grid(True)

In [ ]:
from sklearn.calibration import CalibrationDisplay
disp = CalibrationDisplay.from_predictions(Y_v, y_scores)
plt.grid(True)
plt.show()

In [ ]:
df_val_res["prediction"] = df_val_res["prob_chgoff"].apply(lambda x: 0 if x < THSEL else 1)

In [ ]:
df_val_res.prediction.value_counts()

In [ ]:
df_val_res.LoanStatus.value_counts()

In [ ]:
fp = "../data/sba_loans_prepared/sba_loans_num_enc_test.csv"
df_test = pd.read_csv(fp)

In [ ]:
Xt = df_test[preds]
Yt = df_test.LoanStatus

In [ ]:
test_res = {"prob_chgoff": calibrated_clf.predict_proba(Xt)[:, 1],
                "prob_PIF": calibrated_clf.predict_proba(Xt)[:, 0],
               "LoanStatus": Yt}
df_test_res = pd.DataFrame.from_dict(test_res, orient="columns")

In [ ]:
df_test_res["prediction"] = df_test_res["prob_chgoff"].apply(lambda x: 0 if x < THSEL else 1)

In [ ]:
test_pred_counts = df_test_res.prediction.value_counts()
test_pred_counts[1]/ (test_pred_counts[1] + test_pred_counts[0])

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
print(classification_report(df_test_res.LoanStatus, df_test_res.prediction))

In [ ]:
test_pred_counts[1] + test_pred_counts[0]

In [ ]:
import kmds
from kmds.tagging.tag_types import *
from owlready2 import *
from kmds.ontology.kmds_ontology import *
from kmds.utils.load_utils import *
from kmds.utils.path_utils import *
KNOWLEDGE_BASE = "../data/kmds/sba_loans_kb.xml"
from kmds.ontology.intent_types import IntentType

In [ ]:
onto2 = load_kb(KNOWLEDGE_BASE)
with onto2:
    insts = Workflow.instances()

the_workflow_instance = insts[0]

In [ ]:
mc_obs_list = []
observation_count = 1
sba_modeling_fname = "https://github.com/rajivsam/descriptive_analytics/blob/main/examples/sba_7a_loans_analysis/sba_7a_loans_desc_analysis.pdf"
mc1 = ModellingChoiceObservation(namespace=onto2)
mc1.finding = f"Modeling Plan: Sampling and thresholding are two of the most common approaches to developing a classifier for\
 imbalanced data. The thresholding approach is used here. The probablity of a default is predicted by the model\
 in this case a random forest model. A validation dataset is used to determine the threshold that gives us the\
 desired trade off between precision and recall. Since we need high accuracy with probablity estimate, a probablity\
 calibration model was also used. For details of the modeling, see\n\
 {sba_modeling_fname}."
mc1.finding_sequence = observation_count
mc1.modelling_choice_observation_type = ModellingChoiceTags.MODELLING_CHOICE_OBSERVATION.value
mc1.intent = IntentType.MODEL_EXPLANATION.value
mc_obs_list.append(mc1)


In [ ]:
observation_count += 1
sba_mod_int_fname = "https://github.com/rajivsam/descriptive_analytics/blob/main/examples/sba_7a_loans_analysis/sba_7a_risky_borrower_analysis.pdf"
mc2 = ModellingChoiceObservation(namespace=onto2)
mc2.finding = f"Modeling Plan: When the classifier tags a loan as bad, we need context to understand why it is bad. This\
can be explained by recognizing that bad loans fall in the risky neighborbood (near other bad loans). By clustering the\
risky neighborhood and determining the cluster associated with a loan that is tagged as risky, you can get more context\
on why this loan is determined to be bad by the mode. For details of the clustering and interpretation, see\n\
{sba_mod_int_fname}."
mc2.finding_sequence = observation_count
mc2.modelling_choice_observation_type = ModellingChoiceTags.MODELLING_CHOICE_OBSERVATION.value
mc2.intent = IntentType.MODEL_EXPLANATION.value
mc_obs_list.append(mc2)

In [ ]:
observation_count += 1
mc3 = ModellingChoiceObservation(namespace=onto2)
mc3.finding = f"Modeling Review: (Put in your feedback from model review here.)"
mc3.finding_sequence = observation_count
mc3.modelling_choice_observation_type = ModellingChoiceTags.MODELLING_CHOICE_OBSERVATION.value
mc3.intent = IntentType.MODEL_EXPLANATION.value
mc_obs_list.append(mc3)

In [ ]:
observation_count += 1
mc4 = ModellingChoiceObservation(namespace=onto2)
mc4.finding = f"Modeling Refinement: (1)Refine Feature Engineering of Risky Neighborhood: Add more clusters, use a \
model selection method for clustering to add sufficient risky neighborhood clusters (2) Do feature selection using\
mutual information to select the initial set of attributes."
mc4.finding_sequence = observation_count
mc4.modelling_choice_observation_type = ModellingChoiceTags.MODELLING_CHOICE_OBSERVATION.value
mc4.intent = IntentType.MODEL_EXPLANATION.value
mc_obs_list.append(mc4)

In [ ]:
the_workflow_instance.has_modeling_choice_observations = mc_obs_list
kaw = KnowledgeExtractionExperimentationWorkflow("sba_7a_loans_modeling", namespace=onto)
kaw.has_modeling_choice_observations = mc_obs_list
onto2.save(file=KNOWLEDGE_BASE, format="rdfxml")